# einops-rearrange — worked example 2: Split a channel axis into groups (decomposition with kwarg)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-rearrange`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`rearrange` can **decompose** one axis into two by writing a parenthesized pair on the *input* side: `'b (g c) -> b g c'`. einops cannot guess how to factor the size, so you bind one side with a keyword argument, e.g. `g=n_groups`. This is exactly the reshaping behind GroupNorm, where channels are partitioned into `g` equal groups before normalizing.

## Worked solution

Input is `(b, C)` where the channel count `C` is secretly `g * c` — `g` groups of `c` channels each. We want `(b, g, c)` so each group is its own slice.

**Step 1 — mark the axis to split.** On the left we write `(g c)` in place of the single channel axis. The parentheses on the *input* side mean *this one incoming axis is actually two axes packed together*.

**Step 2 — supply the missing size.** `C` divides into `g*c` in many ways, so einops needs a hint. We pass `g=n_groups`; it then infers `c = C // g`. (We could equally bind `c=...` instead — either one fixes both.)

**Step 3 — name the output.** The right side is `b g c`: keep the batch, then the two freshly-named axes in the order they appeared inside the parens. Row-major again — group index varies slower than within-group channel index, so `g` is the outer split.

**Step 4 — sanity check.** With `C=6`, `g=2`, we expect `c=3`. Channels 0,1,2 land in group 0 and channels 3,4,5 in group 1, which we confirm by comparing slices against the original.

In [ ]:
def split_channels_into_groups(x: Tensor, n_groups: int) -> Tensor:
    return rearrange(x, 'b (g c) -> b g c', g=n_groups)


np.random.seed(0)
t.manual_seed(0)
x = t.arange(2 * 6).reshape(2, 6)
out = split_channels_into_groups(x, n_groups=2)
print('input shape :', tuple(x.shape))
print('output shape:', tuple(out.shape))
print('group 0 of row 0 ==', out[0, 0].tolist(), '(expect channels 0,1,2)')
print('group 1 of row 0 ==', out[0, 1].tolist(), '(expect channels 3,4,5)')